# 00 · Study overview and coverage

> Reference notebook. Every number below is produced by the same functions that
> write the paper's tables (`paper_resources/scripts/`), so this is a readable
> view of the pipeline rather than a second implementation. Regenerate the
> `.tex` files with `python scripts/run_all.py`.


**Question this notebook answers:** what was actually trained and evaluated, and
how does the funnel narrow from the reduced-scale search to the broader regimes?

Backs: Table 1 (study funnel), Table A0 (method coverage), Table A0b (configuration grid).

In [1]:
import sys
from pathlib import Path

import pandas as pd

REPO_ROOT = next(p for p in Path.cwd().resolve().parents if (p / "paper_resources").is_dir())
sys.path.insert(0, str(REPO_ROOT / "paper_resources" / "scripts"))

import common as C

pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 60)
# 4 significant digits rather than 4 decimals, so that a p-value of 5.6e-15 stays
# readable next to an effect size of 0.94.
pd.set_option("display.float_format", lambda v: f"{v:.4g}")

cfg = C.load_config()
BENCHMARKS = cfg["analysis"]["benchmarks"]
ALPHA = cfg["analysis"]["alpha"]
print(f"repo root : {REPO_ROOT}")
print(f"benchmarks: {BENCHMARKS}")
print(f"alpha     : {ALPHA}")

repo root : /data/upb/users/n/nikit/profiles/unix/cs/projects/moe-study-analysis
benchmarks: ['Belebele', 'FLORES', 'SIB200', 'Taxi1500']
alpha     : 0.05


## The two units of counting

The paper counts a **training run** (one configuration trained once) separately
from an **evaluation run** (one model on one benchmark task, where a task is a
benchmark–language pair and the two FLORES directions count separately).
Evaluation coverage is smaller than training coverage at every stage, which is
why both are reported.

In [2]:
import tab0_study_funnel

funnel = pd.read_csv(C.output_dir(cfg, "data") / "tab0_study_funnel.csv")
funnel

,stage,languages,hours,runs,role
0,One-axis rank sweep,10,3.6,47,Effect of rank
1,One-axis expert-count sweep,10,3.6,63,Effect of expert count
2,Mixed follow-up,10,7.2,22,Joint configurations and shortlist
3,Broader validation (96 languages),96,34.5,5,Test the shortlist at scale
4,Broader validation (200 languages),200,72,5,Test the shortlist at scale
5,TOTAL,--,NaN,142,125 unique; 9528 evaluation runs


The funnel is the paper's spine: a wide reduced-scale search on ten languages,
a longer follow-up on the survivors, then five promoted configurations retrained
at 96 and 200 training languages against an unadapted base model.

In [3]:
coverage = pd.read_csv(C.output_dir(cfg, "data") / "tabA0_method_coverage.csv")
coverage

,family,method,rank_settings,rank_runs,component_settings,component_runs,shared_rank_panel,note
0,mola,AdaMoE,2--128,7,2--32,15,True,Complete shared rank panel
1,adamole,AdaMoLE,--,0,2--16,4,False,Higher ranks unstable; $E{=}32$ infeasible
2,hmora,HMoRA,2--128,7,2--32,15,True,Complete shared rank panel
3,hydralora,HydraLoRA,2--256,8,2--32,5,True,Complete shared rank panel
4,lora,LoRA,2--1024,10,--,0,True,"Single adapter, so no expert count"
5,moelora,MOELoRA,2--64,6,2--32,14,True,"Shared rank panel; $E{=}32$, $k{=}16$ infeasible"
6,movlora,MoV,--,1,2--32,5,False,Rank is not defined
7,mtllora,MTL-LoRA,2--256,8,2--32,5,True,Complete shared rank panel


Coverage is deliberately uneven: not every method exposes every design axis
(MoV has no rank to vary, AdaMoLE was not run in the rank search). This is why
RQ2 reports a *shared-panel* comparison of the six methods present at every
rank, rather than one leaderboard over everything.

In [4]:
grid = pd.read_csv(C.output_dir(cfg, "data") / "tabA0b_config_grid.csv")
grid.head(30)

,family,method,rank_configurations,rank_runs,component_configurations,component_runs
0,mola,AdaMoE,"$r \in \{2, 4, 8, 16, 32, 64, 128\}$ at $E{=}8...",7,"$2$:~1; $4$:~1, 2; $8$:~1, 2, 4; $16$:~1, 2, 4...",15
1,adamole,AdaMoLE,--,0,"$2$, $4$, $8$, $16$",4
2,hmora,HMoRA,"$r \in \{2, 4, 8, 16, 32, 64, 128\}$ at $E{=}8...",7,"$2$:~1; $4$:~1, 2; $8$:~1, 2, 4; $16$:~1, 2, 4...",15
3,hydralora,HydraLoRA,"$r \in \{2, 4, 8, 16, 32, 64, 128, 256\}$",8,"$2$, $4$, $8$, $16$, $32$",5
4,lora,LoRA,"$r \in \{2, 4, 8, 16, 32, 64, 128, 256, 512, 1...",10,--,0
5,moelora,MOELoRA,"$r \in \{2, 4, 8, 16, 32, 64\}$ at $E{=}8$, $k...",6,"$2$:~1; $4$:~1, 2; $8$:~1, 2, 4; $16$:~1, 2, 4...",14
6,movlora,MoV,no rank at $E{=}8$,1,"$2$, $4$, $8$, $16$, $32$",5
7,mtllora,MTL-LoRA,"$r \in \{2, 4, 8, 16, 32, 64, 128, 256\}$",8,"$2$, $4$, $8$, $16$, $32$",5


## The five promoted configurations

These are the systems carried into the broader regimes, and the set the RQ3/RQ4
results are about. They were selected on the ten-language follow-up, so the
broader-regime stage tests *these five*, not "the best possible MoE-LoRA".

In [5]:
finalists = pd.DataFrame(cfg["rq3"]["finalists"])
print("baseline:", cfg["rq3"]["baseline_run_id"], "|", cfg["rq3"]["baseline_label"])
finalists

baseline: baseline::l-3-1-8b | Base model


,run_id,followup_run_id,label
0,mtllora_lr5e5_r8_nup16,mtllora_r8_nup16,MTL-LoRA r8/16
1,moelora_tr32_e16_k8,moelora_tr32_e16_k8,MOELoRA r2/16 k8
2,hydralora_r2,hydralora_r2,HydraLoRA r2
3,moelora_tr64_e32_k8,moelora_tr64_e32_k8,MOELoRA r2/32 k8
4,mola_tr16_e8_k6,mola_tr16_e8_k6,AdaMoE r2/8 k6
